# Regularizare și coborârea gradientului — rezolvări minimale


# Obiectivele învățării

- Explicarea funcțiilor de cost, regularizarea, selectarea caracteristicilor și hiperparametrii
- Rezumarea algoritmilor de optimizare statistică complexă, cum ar fi coborârea gradientului și aplicarea sa la regresia liniară



## Introducere

Vom începe cu un scurt tutorial despre regresie, caracteristici polinomiale și regularizare, bazat pe un set de date foarte simplu, rar, care conține o coloană de date `x` și date zgomotoase asociate `y`. Fișierul de date se numește `X_Y_Sinusoid_Data.csv`. 

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import PolynomialFeatures, MinMaxScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV, ElasticNetCV, SGDRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error

## Întrebare 1

* Importați datele. 

* De asemenea, generați aproximativ 100 de puncte de date x egal distanțate pe intervalul 0-1. Folosind aceste puncte, calculați datele y care reprezintă „adevărul de bază” (funcția reală) din ecuația: $y = sin(2\pi x)$

* Reprezentați grafic datele rarefiate (`x` vs `y`) și datele calculate („reale”).


In [ ]:
sinus_path = Path('data/X_Y_Sinusoid_Data.csv')
if sinus_path.exists():
    sinus = pd.read_csv(sinus_path)
    x = sinus.iloc[:, 0].to_numpy(dtype=float)
    y = sinus.iloc[:, 1].to_numpy(dtype=float)
else:
    rng = np.random.default_rng(42)
    x = np.linspace(0, 1, 15)
    y = np.sin(2 * np.pi * x) + rng.normal(0, 0.12, len(x))
    print('CSV-ul sinusoidal lipsește; folosesc date sintetice reproductibile.')
x_grid = np.linspace(0, 1, 100)
y_true = np.sin(2 * np.pi * x_grid)
plt.scatter(x, y, label='Date rarefiate')
plt.plot(x_grid, y_true, label='Funcția reală')
plt.legend(); plt.show()

## Întrebarea 2


* Folosind clasa `PolynomialFeatures` din biblioteca de preprocesare Scikit-learn, creați caracteristici polinomiale de ordinul 20.
* Adaptați aceste date folosind regresia liniară. 
* Reprezentați grafic valoarea prezisă rezultată în comparație cu datele calculate.

Rețineți că `PolynomialFeatures` necesită fie un cadru de date (cu o coloană, nu o serie), fie o matrice 2D de dimensiune (`X`, 1), unde `X` este lungimea.

In [ ]:
poly = PolynomialFeatures(degree=20, include_bias=False)
X_poly = poly.fit_transform(x.reshape(-1, 1))
X_grid_poly = poly.transform(x_grid.reshape(-1, 1))
lr = LinearRegression().fit(X_poly, y)
plt.scatter(x, y, label='Date')
plt.plot(x_grid, y_true, label='Funcția reală')
plt.plot(x_grid, lr.predict(X_grid_poly), label='Gradul 20')
plt.ylim(-2, 2); plt.legend(); plt.show()

## Întrebare 3

* Efectuați regresia pe baza datelor cu caracteristici polinomiale utilizând regresia Ridge ($\alpha$=0,001) și regresia Lasso ($\alpha$=0,0001). 
* Reprezentați grafic rezultatele, așa cum s-a făcut la întrebarea 1. 
* De asemenea, reprezentați grafic magnitudinea coeficienților obținuți din aceste regresii și comparați-i cu cei obținuți din regresia liniară din întrebarea anterioară. Coeficienții regresiei liniare vor necesita probabil un grafic separat (sau propria axă y) din cauza magnitudinii lor mari. 

Ce vă spune magnitudinea comparativ mare a datelor cu privire la rolul regularizării?

In [ ]:
rr = Ridge(alpha=0.001).fit(X_poly, y)
lassor = Lasso(alpha=0.0001, max_iter=50000).fit(X_poly, y)
plt.scatter(x, y, label='Date')
plt.plot(x_grid, y_true, label='Funcția reală')
plt.plot(x_grid, rr.predict(X_grid_poly), label='Ridge')
plt.plot(x_grid, lassor.predict(X_grid_poly), label='Lasso')
plt.legend(); plt.show()

In [ ]:
coefficients = pd.DataFrame({
    'Liniară': np.abs(lr.coef_.ravel()),
    'Ridge': np.abs(rr.coef_.ravel()),
    'Lasso': np.abs(lassor.coef_.ravel()),
})
display(coefficients.describe().loc[['mean', 'max']])
coefficients[['Ridge', 'Lasso']].plot(title='Coeficienți regularizați')
plt.show()
coefficients['Liniară'].plot(title='Coeficienți fără regularizare')
plt.show()

Coeficienții fără regularizare pot deveni foarte mari pentru polinoame de grad înalt. Ridge îi micșorează, iar Lasso poate anula unii coeficienți; astfel se reduce sensibilitatea la zgomot.


## Întrebarea 4

Pentru întrebările rămase, vom lucra cu [setul de date](https://www.kaggle.com/c/house-prices-advanced-regression-techniques) din lecția trecută, care se bazează pe prețurile locuințelor din Ames, Iowa. Există un număr mare de caracteristici - a se vedea exercițiile din săptămâna a treia pentru o discuție despre aceste caracteristici.

Pentru a începe:

* Importați datele cu Pandas, eliminați orice valori nule și o codificare fierbinte a categoriilor. Pot fi utilizate fie codificatoarele de caracteristici Scikit-learn, fie metoda Pandas `get_dummies`.
* Împărțiți datele în seturi de instruire și de testare. 
* Transformarea logaritmică a caracteristicilor distorsionate. 
* Se poate încerca scalarea, deși poate fi interesant să se vadă cât de bine funcționează regularizarea fără scalarea caracteristicilor.

Creați o listă de date categoriale și codificați one-hot. Pandas one-hot encoder (`get_dummies`) funcționează bine cu date care sunt definite ca fiind categoriale.

Apoi, împărțiți datele în seturi de date de instruire și de testare.

Există o serie de coloane care au caracteristici înclinate - li se poate aplica o transformare logaritmică. Rețineți că aceasta include `SalePrice`, predictorul nostru. Cu toate acestea, să o păstrăm așa cum este.

Transformați toate coloanele în care deviația este mai mare de 0,75, cu excepția „SalePrice”.

Separați datele dupa predictor

Date: se folosește `data/train.csv` dacă există; altfel, setul Ames este preluat din OpenML (necesită internet).


In [ ]:
house_path = Path('data/train.csv')
house = pd.read_csv(house_path) if house_path.exists() else fetch_openml(
    name='house_prices', version=1, as_frame=True
).frame.copy()
house = house.drop(columns='Id', errors='ignore').dropna(axis=1)
y_house = pd.to_numeric(house.pop('SalePrice'))
X_house = pd.get_dummies(house, dtype=float)
X_train, X_test, y_train, y_test = train_test_split(
    X_house, y_house, test_size=0.2, random_state=42
)
skewed = X_train.columns[(X_train.skew() > 0.75) &
                         (X_train.min() >= 0) & (X_train.nunique() > 2)]
X_train = X_train.copy()
X_test = X_test.copy()
X_train[skewed] = np.log1p(X_train[skewed])
X_test[skewed] = np.log1p(X_test[skewed])
print(f'{len(X_train)} antrenare, {len(X_test)} test, {X_train.shape[1]} caracteristici')

## Întrebare 5

* Scrieți o funcție **`rmse`** care primește valorile de adevăr și predicție și returnează eroarea medie pătratică. Utilizați funcția `mean_squared_error` de la sklearn.  


* Adaptați un model de regresie liniară de bază
* tipăriți eroarea medie pătratică pentru acest model
* reprezentați grafic prețul de vânzare previzionat față de cel real pe baza modelului.  

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

baseline = LinearRegression().fit(X_train, y_train)
baseline_rmse = rmse(y_test, baseline.predict(X_test))
print(f'RMSE regresie liniară: {baseline_rmse:.2f}')
plt.scatter(y_test, baseline.predict(X_test), alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--')
plt.xlabel('Preț real'); plt.ylabel('Preț prezis'); plt.show()

## Întrebarea 6

Regresia Ridge utilizează normalizarea L2 pentru a reduce magnitudinea coeficienților. Acest lucru poate fi util în situații în care există o varianță ridicată. Funcțiile de regularizare din Scikit-learn conțin fiecare versiuni care au validarea încrucișată încorporată.

* Adaptați un model Ridge regulat (nevalidat încrucișat) la o gamă de valori $\alpha$ și reprezentați grafic RMSE utilizând funcția de eroare validată încrucișat pe care ați creat-o mai sus.
* Utilizați $$[0.005, 0.05, 0.1, 0.3, 1, 3, 5, 10, 15, 30, 80]$$ ca interval de alphas.
* Apoi repetați ajustarea modelelor Ridge folosind gama de valori $\alpha$ din secțiunea anterioară. Comparați rezultatele.

Acum, pentru metoda `RidgeCV`. Din păcate, nu este posibil să obținem valorile alfa pentru modelele care nu au fost selectate. Valorile erorilor și valorile $\alpha$ rezultate sunt foarte asemănătoare cu cele obținute mai sus.

In [ ]:
alphas_ridge = [0.005, 0.05, 0.1, 0.3, 1, 3, 5, 10, 15, 30, 80]
ridge_scores = pd.DataFrame([
    {'alpha': a,
     'RMSE CV': -cross_val_score(Ridge(alpha=a), X_train, y_train,
                                cv=5, scoring='neg_root_mean_squared_error').mean()}
    for a in alphas_ridge
])
ridge_cv = RidgeCV(alphas=alphas_ridge, cv=5,
                   scoring='neg_root_mean_squared_error').fit(X_train, y_train)
ridge_rmse = rmse(y_test, ridge_cv.predict(X_test))
display(ridge_scores)
print(f'RidgeCV: alpha={ridge_cv.alpha_}, RMSE test={ridge_rmse:.2f}')
ridge_scores.plot(x='alpha', y='RMSE CV', logx=True, marker='o')
plt.show()

## Întrebarea 7


La fel ca funcția `RidgeCV`, există și o funcție `LassoCV` care utilizează o funcție de regularizare L1 și validarea încrucișată. Regularizarea L1 va micșora selectiv anumiți coeficienți, efectuând efectiv eliminarea caracteristicilor.

Funcția `LassoCV` nu permite setarea funcției de scor. Cu toate acestea, funcția de eroare personalizată (`rmse`) creată mai sus poate fi utilizată pentru a evalua eroarea modelului final.

În mod similar, există și o funcție elastic net cu validare încrucișată, `ElasticNetCV`, care este o combinație de regularizare L2 și L1.

* Potriviți un model Lasso utilizând validarea încrucișată și determinați valoarea optimă pentru $\alpha$ și RMSE utilizând funcția creată mai sus. Rețineți că magnitudinea lui $\alpha$ poate fi diferită de cea a modelului Ridge.
* Repetați operația cu modelul de plasă elastică.
* Comparați rezultatele prin tabel și/sau grafic.

Utilizați următoarele alphas:

`[1e-5, 5e-5, 0.0001, 0.0005]`

Putem determina câte dintre aceste caracteristici rămân diferite de zero.

Încercați acum plasa elastică, cu aceleași alphas ca în Lasso, și l1_ratios între 0,1 și 0,9

Compararea calculului RMSE din toate modelele este cea mai ușoară într-un tabel.

Putem, de asemenea, să realizăm un grafic al prețurilor reale și previzionate ale locuințelor, la fel ca înainte.

In [ ]:
alphas_sparse = [1e-5, 5e-5, 1e-4, 5e-4]
lasso_cv = LassoCV(alphas=alphas_sparse, cv=5, max_iter=50000,
                   random_state=42).fit(X_train, y_train)
elastic_cv = ElasticNetCV(alphas=alphas_sparse,
                          l1_ratio=np.arange(0.1, 1, 0.2), cv=5,
                          max_iter=50000, random_state=42).fit(X_train, y_train)
results = pd.DataFrame([
    {'Model': 'Liniară', 'alpha': np.nan, 'RMSE test': baseline_rmse,
     'Coeficienți nenuli': np.count_nonzero(baseline.coef_)},
    {'Model': 'RidgeCV', 'alpha': ridge_cv.alpha_, 'RMSE test': ridge_rmse,
     'Coeficienți nenuli': np.count_nonzero(ridge_cv.coef_)},
    {'Model': 'LassoCV', 'alpha': lasso_cv.alpha_,
     'RMSE test': rmse(y_test, lasso_cv.predict(X_test)),
     'Coeficienți nenuli': np.count_nonzero(lasso_cv.coef_)},
    {'Model': 'ElasticNetCV', 'alpha': elastic_cv.alpha_,
     'RMSE test': rmse(y_test, elastic_cv.predict(X_test)),
     'Coeficienți nenuli': np.count_nonzero(elastic_cv.coef_)},
])
display(results)
print(f'ElasticNet l1_ratio={elastic_cv.l1_ratio_}')
plt.scatter(y_test, elastic_cv.predict(X_test), alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--')
plt.xlabel('Preț real'); plt.ylabel('Preț prezis (ElasticNet)'); plt.show()

## Întrebarea 8

Să explorăm coborârea stochastică a gradientului în acest exercițiu.  
Reamintim că modelele liniare în general sunt sensibile la scalare.
Cu toate acestea, SGD este *foarte* sensibil la scalare.  
În plus, o valoare mare a ratei de învățare poate determina algoritmul să devieze, în timp ce o valoare prea mică poate dura prea mult până la convergență.

* Potriviți un model stochastic gradient descent fără o penalizare de regularizare (parametrul relevant este `penalty`).
* Ajustați acum modele stochastice de coborâre în gradient cu fiecare dintre cele trei penalizări (L2, L1, Elastic Net) folosind valorile parametrilor determinate prin validarea încrucișată de mai sus. 
* Nu scalați datele înainte de ajustarea modelului.  
* Comparați rezultatele cu cele obținute fără a utiliza coborârea stochastică în gradient.

In [ ]:
sgd_params = {
    'Fără penalizare': {'penalty': None},
    'L2': {'penalty': 'l2', 'alpha': ridge_cv.alpha_},
    'L1': {'penalty': 'l1', 'alpha': lasso_cv.alpha_},
    'Elastic Net': {'penalty': 'elasticnet', 'alpha': elastic_cv.alpha_,
                    'l1_ratio': elastic_cv.l1_ratio_},
}
def fit_sgd(X_fit, X_eval, eta0=None):
    rows = []
    for name, params in sgd_params.items():
        options = {'random_state': 42, 'max_iter': 2000, **params}
        if eta0 is not None:
            options['eta0'] = eta0
        model = SGDRegressor(**options).fit(X_fit, y_train)
        rows.append({'Model': name, 'RMSE test': rmse(y_test, model.predict(X_eval))})
    return pd.DataFrame(rows)

sgd_unscaled = fit_sgd(X_train, X_test)
display(sgd_unscaled)
display(results[['Model', 'RMSE test']])

Observați cât de mari sunt valorile erorii! Algoritmul este divergent. Acest lucru se poate datora faptului că scalarea și/sau rata de învățare sunt prea mari. Să ajustăm rata de învățare și să vedem ce se întâmplă.

* Introduceți `eta0=1e-7` la crearea instanței de `SGDClassifier`.
* Calculați din nou erorile pentru toate penalitățile și comparați.

In [ ]:
sgd_small_lr = fit_sgd(X_train, X_test, eta0=1e-7)
display(sgd_small_lr)

Acum să ne scalăm datele de antrenament și să încercăm din nou.

* Adaptați un `MinMaxScaler` la `X_train` și creați o variabilă `X_train_scaled`.
* Folosind scala, transformați `X_test` și creați o variabilă `X_test_scaled`. 
* Aplicați-le aceleași versiuni ale SGD și comparați rezultatele. De data aceasta nu introduceți un eta0.

In [ ]:
scaler = MinMaxScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
sgd_scaled = fit_sgd(X_train_scaled, X_test_scaled)
comparison = sgd_unscaled.rename(columns={'RMSE test': 'Fără scalare'})
comparison['eta0=1e-7'] = sgd_small_lr['RMSE test']
comparison['Cu scalare'] = sgd_scaled['RMSE test']
display(comparison)